# PPO on Splendor

Reference: `rl/examples/ppo/lunar_lander.ipynb`

In [ ]:
import numpy as np
import torch as t
import torch.nn.functional as F
from einops import rearrange
from tqdm import tqdm

import gymnasium as gym
import rl.splendor  # registers Splendor-v0
from rl.ppo import PPO, split_obs

In [ ]:
# Initialise the environment
n_envs = 8
n_updates = 2000
n_rollout = 128
device = "cuda" if t.cuda.is_available() else "cpu"

envs = gym.make_vec("Splendor-v0", num_envs=n_envs, max_turns=200)

envs_wrapper = gym.wrappers.vector.RecordEpisodeStatistics(
  envs, buffer_length=n_envs*n_updates
)

obs_shape = int(envs.single_observation_space["obs"].shape[0])
action_shape = int(envs.single_action_space.n)
obs_shape, action_shape

In [ ]:
ppo = PPO(n_actions=action_shape, d_state=obs_shape, device=device).to(device)
df = 0.99
lam = 0.95
ent_coef = 0.01
n_epochs = 5
mb_size = 256
eps = 0.2

obs_dict, info = envs_wrapper.reset(seed=0)
obs, mask = split_obs(obs_dict, device)

was_done = t.zeros(n_envs, dtype=t.bool, device=device)

pbar = tqdm(range(n_updates))
for update in pbar:
  ep_obs = t.zeros(n_rollout, n_envs, obs_shape, device=device)
  ep_masks = t.zeros(n_rollout, n_envs, action_shape, dtype=t.bool, device=device)
  ep_state_values = t.zeros(n_rollout, n_envs, device=device)
  ep_actions = t.zeros(n_rollout, n_envs, device=device, dtype=t.long)
  ep_action_log_probs = t.zeros(n_rollout, n_envs, device=device)
  ep_rewards = t.zeros(n_rollout, n_envs, device=device)
  masks = t.zeros(n_rollout, n_envs, device=device)
  valid = t.zeros(n_rollout, n_envs, device=device)

  with t.no_grad():
    for step in range(n_rollout):
      actions, action_log_probs, state_values = ppo.select_action(obs, mask)

      states, rewards, terminated, truncated, infos = envs_wrapper.step(actions.cpu().numpy())
      done = terminated | truncated

      ep_obs[step] = obs
      ep_masks[step] = mask
      ep_state_values[step] = state_values.squeeze()
      ep_rewards[step] = t.tensor(rewards, device=device)
      ep_actions[step] = actions
      ep_action_log_probs[step] = action_log_probs

      valid[step] = (~was_done).float()
      masks[step] = t.tensor(~done, dtype=t.float32, device=device) * valid[step]

      obs, mask = split_obs(states, device)
      was_done = t.tensor(done, dtype=t.bool, device=device)

      # true final obs
      if truncated.any():
        ep_rewards[step] += df * ppo.critic(obs).squeeze(-1) * t.tensor(truncated, dtype=t.float32, device=device)

    v_T = ppo.critic(obs).squeeze(-1)

  advantages = t.zeros(n_rollout, n_envs, device=device)
  next_value, next_adv = v_T, 0
  for i in reversed(range(n_rollout)):
    delta = ep_rewards[i] + df * next_value * masks[i] - ep_state_values[i]
    advantages[i] = delta + df * lam * next_adv * masks[i]
    next_value, next_adv = ep_state_values[i], advantages[i]

  # flatten & drop dummy steps
  b_valid = rearrange(valid, "nr ne -> (nr ne)").bool()
  b_obs = rearrange(ep_obs, "nr ne d -> (nr ne) d")[b_valid]
  b_masks = rearrange(ep_masks, "nr ne a -> (nr ne) a")[b_valid]
  b_actions = rearrange(ep_actions, "nr ne -> (nr ne)")[b_valid]
  b_action_log_probs = rearrange(ep_action_log_probs, "nr ne -> (nr ne)")[b_valid]
  b_state_values = rearrange(ep_state_values, "nr ne -> (nr ne)")[b_valid]
  b_advantages = rearrange(advantages, "nr ne -> (nr ne)")[b_valid]
  b_returns  = b_advantages + b_state_values

  batch_size = int(b_valid.sum())
  b_inds = np.arange(batch_size)

  for epoch in range(n_epochs):
    np.random.shuffle(b_inds)

    for start in range(0, batch_size, mb_size):
      mb = b_inds[start:start+mb_size]

      mb_adv = b_advantages[mb]
      mb_adv = (mb_adv - mb_adv.mean()) / (mb_adv.std() + 1e-8)

      new_log_policy, new_state_values, entropy = ppo.evaluate(b_obs[mb], b_actions[mb], b_masks[mb])
      r = t.exp(new_log_policy - b_action_log_probs[mb])

      unclipped = r*mb_adv
      clipped   = t.clamp(r, 1 - eps, 1 + eps) * mb_adv

      policy_loss = -t.min(unclipped, clipped).mean() - ent_coef * entropy.mean()
      critic_loss = F.mse_loss(new_state_values.squeeze(), b_returns[mb])

      ppo.update_params(policy_loss, critic_loss)

      if len(envs_wrapper.return_queue) > 0:
        recent = np.array(envs_wrapper.return_queue)[-100:]
        pbar.set_postfix(
          ret=f"{recent.mean():.2f}",
          len=f"{np.array(envs_wrapper.length_queue)[-100:].mean():.0f}",
          ploss=f"{policy_loss.item():.3f}",
          closs=f"{critic_loss.item():.3f}",
          refresh=False,
        )

## Evaluation

A random policy wins ~48%

In [ ]:
from rl.splendor import SplendorEnv

def win_rate(ppo, episodes=200, seed=10_000, greedy=True):
  env = SplendorEnv()
  wins = draws = 0
  lengths = []
  for ep in range(episodes):
    obs_dict, _ = env.reset(seed=seed + ep)
    done = False
    steps = 0
    while not done:
      obs, mask = split_obs({k: v[None] for k, v in obs_dict.items()}, ppo.device)
      with t.no_grad():
        logits, _ = ppo.forward(obs, mask)
        action = int(logits.argmax(-1)) if greedy else int(t.distributions.Categorical(logits=logits).sample())
      obs_dict, reward, terminated, truncated, _ = env.step(action)
      steps += 1
      done = terminated or truncated
    wins += reward > 0
    draws += reward == 0
    lengths.append(steps)
  return wins / episodes, draws / episodes, float(np.mean(lengths))

w, d, l = win_rate(ppo)
print(f"win {w:.1%} | draw {d:.1%} | mean turns {l:.0f}")

In [ ]:
import matplotlib.pyplot as plt

returns = np.array(envs_wrapper.return_queue).squeeze()
window = 500
smoothed = np.convolve(returns, np.ones(window)/window, mode="valid")

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(smoothed)
ax.set_xlabel("episode")
ax.set_ylabel(f"return (mean of {window})")
ax.axhline(0, color="k", lw=0.5)
ax.set_title("PPO on Splendor vs random opponent")
plt.show()